# Git 标准使用流程（面向日常开发）

> 文件名：`git_usage.ipynb`  
> 目标：当你在项目里新增文件夹和代码文件时，按一套稳定、可复用的标准流程执行。

---

## 0. 你需要记住的核心原则

1. **编辑器负责写代码，Git 负责管理历史和协作**。
2. **不要直接在 `main/master` 上开发**。
3. **每次 commit 只做一件事**（小而清晰）。
4. **先同步再开发，先自检再提交**。
5. **提交信息可读、可追踪**。


## 1. 一次性初始化（只做一次）

如果你还没配置 Git 身份：

```bash
git config --global user.name "你的名字"
git config --global user.email "你的邮箱"
```

建议设置默认 pull 策略为 rebase（减少无意义 merge commit）：

```bash
git config --global pull.rebase true
git config --global rebase.autoStash true
```

检查配置：

```bash
git config --global --list
```


## 2. 日常标准流程（创建新文件夹 + 新代码文件）

下面是你最常用的模板流程。


### 步骤 A：切到主分支并更新

```bash
git switch main
git pull --rebase
```

如果你的主分支名是 `master`，把 `main` 替换为 `master`。


### 步骤 B：创建功能分支

```bash
git switch -c feature/add-data-loader
```

分支命名建议：
- `feature/<功能>`
- `fix/<问题>`
- `chore/<维护项>`


### 步骤 C：创建目录和文件

你可以在 VS Code 里创建；命令行等价如下：

```bash
mkdir -p src/data_loader
touch src/data_loader/__init__.py
touch src/data_loader/loader.py
touch tests/test_loader.py
```

说明：
- `mkdir -p`：目录不存在就创建，存在也不报错。
- `touch`：创建空文件（如果已存在则更新时间戳）。


In [ ]:
# 可执行示例：在当前仓库创建一个 demo 模块（运行前先确认路径）
# !mkdir -p src/data_loader
# !touch src/data_loader/__init__.py src/data_loader/loader.py tests/test_loader.py
# !find src -maxdepth 3 -type f | sort


### 步骤 D：写入最小可运行内容

避免只提交空文件，至少写入基础逻辑。

`src/data_loader/loader.py` 示例：

```python
from pathlib import Path
import csv


def load_csv(path: str):
    """Load CSV rows into a list of dicts."""
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"File not found: {path}")

    with p.open("r", encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))
```


### 步骤 E：本地自检（非常关键）

在提交前运行测试/检查：

```bash
# Python 示例
pytest -q

# 或者至少做语法检查
python -m py_compile src/data_loader/loader.py
```

如果项目是 Node/Java/Go，请替换为对应命令（如 `npm test`、`mvn test`、`go test ./...`）。


### 步骤 F：查看改动并暂存

```bash
git status
git diff

# 只暂存本次要提交的文件（推荐精确 add）
git add src/data_loader/__init__.py src/data_loader/loader.py tests/test_loader.py

# 再看一次暂存区改动
git diff --staged
```


### 步骤 G：提交（Commit）

提交信息建议采用 `type: summary`：

```bash
git commit -m "feat: add CSV data loader and tests"
```

常见 type：
- `feat`：新功能
- `fix`：修复
- `docs`：文档
- `refactor`：重构
- `test`：测试
- `chore`：杂项维护


### 步骤 H：推送并发起 PR

```bash
git push -u origin feature/add-data-loader
```

然后在 GitHub/GitLab 发起 Pull Request（或 Merge Request）到 `main`。


## 3. 高频问题与处理

### 3.1 我改错了文件，还没 commit

```bash
# 取消暂存
git restore --staged <file>

# 丢弃工作区修改（谨慎）
git restore <file>
```

### 3.2 当前分支很脏，但我想先切走

```bash
git stash
git switch main
# 回来后
git stash pop
```

### 3.3 想快速看清历史

```bash
git log --oneline --graph --decorate --all
```


## 4. 推荐的 .gitignore（Python 项目示例）

```gitignore
__pycache__/
*.py[cod]
.pytest_cache/
.mypy_cache/
.venv/
venv/
.env
dist/
build/
*.log
```

注意：密钥、token、证书永远不要提交到仓库。


## 5. 一套可直接复用的命令模板

```bash
# 0) 同步主分支
git switch main
git pull --rebase

# 1) 新建功能分支
git switch -c feature/<name>

# 2) 建目录和文件
mkdir -p <dir>
touch <dir>/<file1> <dir>/<file2>

# 3) 开发 + 自测
# (run tests / build)

# 4) 检查并提交
git status
git diff
git add <files>
git diff --staged
git commit -m "feat: <summary>"

# 5) 推送
git push -u origin feature/<name>
```


## 6. 你当前场景的结论

你继续使用 VS Code 建文件完全可以，**这不影响专业性**。

真正的标准做法是：
- 用 VS Code 编写和管理文件结构
- 用 Git 做版本控制、分支协作、可追踪提交
- 需要批量或自动化时，再配合 shell 命令

按本 Notebook 的流程执行，你的开发习惯就很规范。
